In [ ]:
from tracemalloc import start
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

class HeartbeatDataProcessor:
    def __init__(self, folder_path, filtered_df_path,window_size=2, step_size=1,boundary_cut=5):
            """
            Initializes the data pipeline reader.
            
            Parameters:
            - file_path (str): Path to the df_intervals CSV data.
            - window_size (int): Number of consecutive intervals per training chunk.
            - step_size (int): How far the window slides forward (enables overlapping).
            """
            self.folder_path =   folder_path
            self.filtered_df_path = filtered_df_path
            self.window_size = window_size
            self.step_size = step_size
            self.boundary_cut = boundary_cut
            # Initialize internal storage and stateful scaler
            self.df_filtered_raw = None
            self.filtered_index = None
            self.scaler = StandardScaler()
            self.subject_segment_dict = {}

    def load_filtered_df(self,subject_num):
        #the data here already filtered by activity id!=0 and length>20
        filtered_df_path = f"{self.filtered_df_path}filtered_activities.csv"
        self.filtered_index = pd.read_csv(filtered_df_path)

    def preprocess_subjects(self,subject_array):
        # 1. Ensure the filtered index is loaded
        if self.filtered_index is None:
            self.load_filtered_df(subject_array)

        # 2. Preprocess each subject
        for subject_num in subject_array:
            file_path = f"{self.folder_path}subject{subject_num}.dat"
            df_raw = pd.read_csv(file_path, sep='\s+', header=None)
            subject_intervals = self.filtered_index[self.filtered_index['subject_id'] == subject_num]

            self.subject_segment_dict[subject_num] = []

            for interval_id, interval in subject_intervals.iterrows():
                start_t = int(interval['t1'])+self.boundary_cut
                end_t = int(interval['t2'])-self.boundary_cut

                #segment by window_size and step_size
                start_dt_array = np.arange(start_t,end_t-self.window_size,self.step_size)

                for start_dt in start_dt_array:
                    # 3. Extract the interval from the raw data
                    chunk = df_raw[(df_raw[0]>=start_dt) & (df_raw[0]<=start_dt+self.window_size)].copy()
                    if not chunk.empty:
                        chunk['subject_id'] = subject_num
                        chunk['interval_id'] = interval_id

                    self.subject_segment_dict[subject_num].append(chunk)
            print(self.subject_segment_dict[subject_num][0])
        
        # 4. Combine all chunks into a single DataFrame
        self.df_filtered  = pd.concat(self.subject_segment_dict[subject_num],ignore_index=True)

        # 5. Standardize the features
        # df_combined[['x', 'y', 'z']] = self.scaler.fit_transform(df_combined[['x', 'y', 'z']])

        def extract_features(self,window_df):
            # Extract features from the windowed data
            return self


In [ ]:
folder_path='../data/PAMAP2_Dataset/Protocol/'
filtered_df_path='./'
processed_data = HeartbeatDataProcessor(folder_path,filtered_df_path)
processed_data.preprocess_subjects(range(101,108))
# print(processed_data.df_filtered.head(3))
# print(processed_data.filtered_index.head(3))